# Embeddings to Classification

# Imports

In [28]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.calibration import calibration_curve

# Data Manipulation

In [29]:
X = np.load("/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/all_embeddings.npy")
df = pd.read_csv("/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/embedding_order.csv")
labels = df["BBclass"].astype(str)
mask = ~labels.isna() & (labels.str.lower() != "nan") & (labels.str.strip() != "")
labels = labels[mask]
X = X[mask]
le = LabelEncoder()
y = le.fit_transform(labels)

# Train Test Split

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Model Training and Predictions

In [31]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist"
)
model.fit(X_train, y_train)

#predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evalutaion Metrics

In [32]:

plot_dir = "/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/Embeddings_Plots"
os.makedirs(plot_dir, exist_ok=True)

print("\n=== MODEL PERFORMANCE ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


# Confusion Matrix

plt.figure(figsize=(6,5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title("Confusion Matrix", fontsize=14)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "confusion_matrix.png"), dpi=300)
plt.close()


#ROC Curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, linewidth=2)
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve", fontsize=14)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "roc_curve.png"), dpi=300)
plt.close()


#Precision–Recall Curve

prec, recall, _ = precision_recall_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(recall, prec, linewidth=2)
plt.title("Precision–Recall Curve", fontsize=14)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "precision_recall_curve.png"), dpi=300)
plt.close()


#Feature Importance (XGBoost)

plt.figure(figsize=(8,6))
xgb.plot_importance(model, max_num_features=20)
plt.title("Top 20 Feature Importances", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "feature_importance.png"), dpi=300)
plt.close()

#Calibration Curve

prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
plt.figure(figsize=(6,5))
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1], [0,1], '--')
plt.title("Calibration Curve", fontsize=14)
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "calibration_curve.png"), dpi=300)
plt.close()

print(f"\nAll plots saved in '{plot_dir}' directory")



=== MODEL PERFORMANCE ===
Accuracy: 0.7375930521091811
F1 Score: 0.8300522298111691
ROC-AUC: 0.7729503606214951

Classification Report:

              precision    recall  f1-score   support

        BBB+       0.56      0.34      0.42       458
        BBB-       0.77      0.90      0.83      1154

    accuracy                           0.74      1612
   macro avg       0.67      0.62      0.63      1612
weighted avg       0.71      0.74      0.71      1612


All plots saved in '/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/Embeddings_Plots' directory


<Figure size 800x600 with 0 Axes>

# logBBB prediction

# Imports

In [33]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier
import xgboost as xgb

# Data Manipulation

In [34]:

df = pd.read_csv("/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/combined_compound_SMILES_BBclass_with_FP.csv")
fp_cols = [f"fp_{i}" for i in range(2048)]
X_fp = df[fp_cols].values
y_logBB = df["logBB"].values
labels = df["BBclass"].astype(str)

# handling infinity values
mask = ~pd.isna(y_logBB) & np.isfinite(y_logBB)
X_fp = X_fp[mask]
y_logBB = y_logBB[mask]
labels = labels[mask]

# Only BBB+ / BBB- for classification
class_mask = labels.isin(["BBB+", "BBB-"])
X_fp_class = X_fp[class_mask]
y_class = labels[class_mask]

# Encode BBclass
le = LabelEncoder()
y_class = le.fit_transform(y_class)

print("After cleaning:")
print("Regression shape:", X_fp.shape, y_logBB.shape)
print("Classification shape:", X_fp_class.shape, y_class.shape)
print("BBclass mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

/tmp/ipython-input-3817842026.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/combined_compound_SMILES_BBclass_with_FP.csv")


After cleaning:
Regression shape: (1218, 2048) (1218,)
Classification shape: (718, 2048) (718,)
BBclass mapping: {'BBB+': np.int64(0), 'BBB-': np.int64(1)}


# Train Test Split

In [35]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_fp, y_logBB, test_size=0.2, random_state=42
)

# Model Training, Prediction and Evaluation of logBBB

In [38]:
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train_reg, y_train_reg)

# Predict and evaluate
y_pred_reg = rf_model.predict(X_test_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)
print("\n=== logBB Regression (RandomForest) ===")
print("R2:", r2)
print("RMSE:", rmse)

plot_dir = "/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/Plots_Morgan"
os.makedirs(plot_dir, exist_ok=True)

# Plot Actual vs Predicted logBB
plt.figure(figsize=(6,6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()], 'r--')
plt.xlabel("Actual logBB")
plt.ylabel("Predicted logBB")
plt.title("logBB Regression: Actual vs Predicted")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "logBB_regression.png"), dpi=300)
plt.close()


=== logBB Regression (RandomForest) ===
R2: 0.11276333309151032
RMSE: 0.7316844659000211


# Morgan FP + logBBB for classification

# Features

In [39]:
# Predicted logBB for all compounds
predicted_logBB = rf_model.predict(X_fp_class).reshape(-1,1)

# Combine fingerprints + predicted logBB
X_class_combined = np.hstack([X_fp_class, predicted_logBB])

# Train Test Split

In [40]:
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class_combined, y_class, test_size=0.2, random_state=42, stratify=y_class
)

print("Classification train/test shapes:", X_train_class.shape, X_test_class.shape)

Classification train/test shapes: (574, 2049) (144, 2049)


# Model Training and Prediction

In [41]:
clf_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1
)
clf_model.fit(X_train_class, y_train_class)

#Predictions & Metrics

y_pred_class = clf_model.predict(X_test_class)
y_prob_class = clf_model.predict_proba(X_test_class)[:,1]

# Evalution Metrics

In [42]:

plot_dir = "/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/Plots_Morgan"
os.makedirs(plot_dir, exist_ok=True)

print("\n=== BBclass Classification Performance ===")
print("Accuracy:", accuracy_score(y_test_class, y_pred_class))
print("F1 Score:", f1_score(y_test_class, y_pred_class))
print("ROC-AUC:", roc_auc_score(y_test_class, y_prob_class))
print("\nClassification Report:\n")
print(classification_report(y_test_class, y_pred_class, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test_class, y_pred_class)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "BBclass_confusion.png"), dpi=300)
plt.close()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_class, y_prob_class)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("BBclass ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "BBclass_roc_curve.png"), dpi=300)
plt.close()

# Precision-Recall Curve
prec, recall, _ = precision_recall_curve(y_test_class, y_prob_class)
plt.figure(figsize=(6,5))
plt.plot(recall, prec)
plt.title("BBclass Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "BBclass_precision_recall.png"), dpi=300)
plt.close()

# Feature Importance
plt.figure(figsize=(8,6))
xgb.plot_importance(clf_model, max_num_features=20)
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "BBclass_feature_importance.png"), dpi=300)
plt.close()

print(f"\nAll plots saved in '{plot_dir}' directory")



=== BBclass Classification Performance ===
Accuracy: 0.9652777777777778
F1 Score: 0.8275862068965517
ROC-AUC: 0.978860833822666

Classification Report:

              precision    recall  f1-score   support

        BBB+       0.99      0.97      0.98       131
        BBB-       0.75      0.92      0.83        13

    accuracy                           0.97       144
   macro avg       0.87      0.95      0.90       144
weighted avg       0.97      0.97      0.97       144


All plots saved in '/content/drive/MyDrive/Chemi_Project/Model_Training and Evaluation/Plots_Morgan' directory


<Figure size 800x600 with 0 Axes>